In [ ]:
# Model Notebook
# This notebook trains a Random Forest classifier to detect fraud using the cleaned dataset exported from the preprocessing notebook.

import pandas as pd
# allows us to load the cleaned CSV and work with it as a DataFrame

from sklearn.model_selection import train_test_split, RandomizedSearchCV
# train_test_split divides our data into training and testing sets
# RandomizedSearchCV searches over a range of hyperparameters more efficiently than GridSearchCV

from sklearn.ensemble import RandomForestClassifier
# the classifier we're using to detect fraud

from sklearn.metrics import f1_score, classification_report
# f1_score balances precision and recall, which matters here given the severe class imbalance
# classification_report gives us a fuller breakdown (precision, recall, f1, support) per class



In [ ]:
fraud_clean = pd.read_csv('../data/cleaned_fraud.csv')
# loads the cleaned, feature engineered dataset exported from the cleaning notebook

In [ ]:
# Splits the cleaned dataset into features (X) and the target (y), excluding isFraud from the input features since that's what we're predicting

X = fraud_clean.drop(columns=['isFraud'])
# X holds every column except the target these are our model's input features

y = fraud_clean['isFraud']
# y holds only the target column what we're trying to predict

In [ ]:
#Splits the data into 80% training and 20% testing sets. stratify=y ensures both sets preserve the same fraud/non-fraud ratio, which matters given how rare fraud is in this dataset (0.13%).

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
# splits data into 80% train / 20% test
# random_state=42 makes the split reproducible
# stratify=y preserves the same fraud/non fraud ratio in both train and test sets, critical given how rare fraud is

In [ ]:
baseline_model = RandomForestClassifier(random_state=42, n_jobs=-1)
# creates a baseline Random Forest with default settings
# n_jobs=-1 uses all available CPU cores to speed up training on this large dataset

baseline_model.fit(X_train, y_train)
# trains the baseline model on the training data

RandomForestClassifier(n_jobs=-1, random_state=42)

Instantiating a baseline RandomForestClassifier using scikit-learn's default hyperparameters (100 trees, no maximum depth limit), with random_state=42 for reproducibility and n_jobs=-1 to use all available CPU cores. This model will be trained as is, without any tuning, to establish a performance baseline before running hyperparameter search

In [7]:
baseline_preds = baseline_model.predict(X_test)
# generates predictions on the unseen test set
print("Baseline F1 Score:", f1_score(y_test, baseline_preds))
# prints the F1 score for the untuned baseline model, our starting point for comparison
print(classification_report(y_test, baseline_preds))
# prints precision, recall, and F1 broken down by class, so we can see how well fraud (class 1) specifically is detected

Baseline F1 Score: 0.8762160348876216
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.98      0.79      0.88      1643

    accuracy                           1.00   1272524
   macro avg       0.99      0.90      0.94   1272524
weighted avg       1.00      1.00      1.00   1272524



Baseline Random Forest results: F1 score of 0.8762 for the fraud class which is good overall; with a precision of 0.98 and a recall of 0.79 the model leans more towards precision(not false accusing of fraud) but also does a solid job of catching the majority of fraud (79 out of 100 cases). This serves as the untuned starting point before hyperparameter search.

In [10]:
X_train_sample = X_train.sample(n=300000, random_state=42)
# heavily samples the training data just for the search step, to keep fit times short

y_train_sample = y_train.loc[X_train_sample.index]
# pulls matching labels for the sampled rows

In [ ]:
param_distributions = {
    'n_estimators': [50, 75],
    'max_depth': [8, 12]
}
# a small, fast to search grid fewer trees, capped depth

In [ ]:
random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=param_distributions,
    n_iter=2, cv=2, scoring='f1', random_state=42, n_jobs=-1)
# only 4 total fits (2 combos × 2 folds), each on a smaller sample

random_search.fit(X_train_sample, y_train_sample)
# fits the search; by default, RandomizedSearchCV automatically retrains the best combo


RandomizedSearchCV(cv=2,
                   estimator=RandomForestClassifier(n_jobs=-1, random_state=42),
                   n_iter=2, n_jobs=-1,
                   param_distributions={'max_depth': [8, 12],
                                        'n_estimators': [50, 75]},
                   random_state=42, scoring='f1')

The best_estimator_ confirms the winning RandomForestClassifier configuration selected by RandomizedSearchCV.

In [13]:
print("Best hyperparameters found:", random_search.best_params_)
# prints which combination of hyperparameters performed best during the search

best_model = random_search.best_estimator_
# extracts the best performing model found during the search, already retrained on the full training set

Best hyperparameters found: {'n_estimators': 75, 'max_depth': 12}


Hyperparameter search selected 75 trees with a maximum depth of 12 as the best configuration, evaluated on a 300,000 row sample of the training data to keep search time manageable. Compared to the baseline's defaults (100 trees, unlimited depth), this is a notably smaller and shallower model.

In [14]:
tuned_preds = best_model.predict(X_test)
# generates predictions on the test set using the tuned model

print("Tuned F1 Score:", f1_score(y_test, tuned_preds))
# prints the F1 score for the tuned model, to compare against the baseline

print(classification_report(y_test, tuned_preds))
# prints the full precision/recall/F1 breakdown for the tuned model

Tuned F1 Score: 0.8423969518531348
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.98      0.74      0.84      1643

    accuracy                           1.00   1272524
   macro avg       0.99      0.87      0.92   1272524
weighted avg       1.00      1.00      1.00   1272524



Exception ignored in: <function ResourceTracker.__del__ at 0x10c59db20>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x110b5db20>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x10218db20>
Traceback (most recent call last

Tuned Random Forest results: F1 score of 0.8423 for the fraud class which is good overall but worse than the baseline (0.8762); precision of 0.98 (same as baseline) and a recall of 0.74 ( also worse than baseline 0.79) the model leans more towards precision (not false accusing of fraud) while also doing a solid job of catching the majority of fraud (74 out of 100 cases). But the tuned model performed slightly worse than the baseline likely because the hyperparameter search was run on a 300,000 row sample of the training data rather than the full ~6.36 million rows. The smaller sample and capped depth may not have captured the full complexity the baseline's larger, unrestricted model was able to learn from.